#Import Library

In [2]:
import pandas as pd
import numpy as np

#Import Dataset

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/HazemNegm/GDG-Final-Project/968668ebe662b3d9f3ef228b7b6348d8cc60995c/Raw%20Data/global_tech_startups_2026.csv")

#Dim_Location

In [4]:
dim_location = df[['Country', 'City']].drop_duplicates().reset_index(drop=True)
dim_location['Location_ID'] = dim_location.index + 1

#Dim_Domain

In [5]:
dim_domain = df[['Domain']].drop_duplicates().reset_index(drop=True)
dim_domain['Domain_ID'] = dim_domain.index + 1

#Dim_Time

In [6]:
dim_time = df[['Founding_Year']].drop_duplicates().reset_index(drop=True)
dim_time['Time_ID'] = dim_time.index + 1

#Dim_Status

In [7]:
dim_status = df[['Funding_Stage', 'Investor_Tier', 'Acquisition_Status', 'AI_Adoption_Level']].drop_duplicates().reset_index(drop=True)
dim_status['Status_ID'] = dim_status.index + 1

#Fact Table

In [8]:
fact_table = df.copy()

# Engineer Key Financial Metrics for Analysis
fact_table['Revenue_Per_Employee_Millions'] = fact_table['Revenue_ARR_Millions'] / fact_table['Current_Headcount_2026'].replace(0, np.nan)
fact_table['Valuation_to_ARR_Multiple'] = fact_table['Valuation_USD_Millions'] / fact_table['Revenue_ARR_Millions'].replace(0, np.nan)
fact_table['Annualized_Burn_Rate_Millions'] = fact_table['Monthly_Burn_Rate_Millions'] * 12

# Merge Foreign Keys
fact_table = fact_table.merge(dim_location, on=['Country', 'City'], how='left')
fact_table = fact_table.merge(dim_domain, on=['Domain'], how='left')
fact_table = fact_table.merge(dim_time, on=['Founding_Year'], how='left')
fact_table = fact_table.merge(dim_status, on=['Funding_Stage', 'Investor_Tier', 'Acquisition_Status', 'AI_Adoption_Level'], how='left')

# Isolate the Final Fact Table
fact_cols = [
    'Company_ID', 'Location_ID', 'Domain_ID', 'Time_ID', 'Status_ID',
    'Total_Funding_USD_Millions', 'Valuation_USD_Millions', 'Revenue_ARR_Millions',
    'Monthly_Burn_Rate_Millions', 'Annualized_Burn_Rate_Millions', 'Runway_Months_2024',
    'Peak_Headcount_2023', 'Layoffs_2024_2025', 'Current_Headcount_2026',
    'Revenue_Per_Employee_Millions', 'Valuation_to_ARR_Multiple'
]
fact_company_financials = fact_table[fact_cols]

#Download Data

In [9]:
import os
export_folder = 'Star_Schema_Export'
os.makedirs(export_folder, exist_ok=True)

# Export Dimensions
dim_location.to_csv(f'{export_folder}/Dim_Location.csv', index=False)
dim_domain.to_csv(f'{export_folder}/Dim_Domain.csv', index=False)
dim_time.to_csv(f'{export_folder}/Dim_Time.csv', index=False)
dim_status.to_csv(f'{export_folder}/Dim_Status.csv', index=False)

# Export Fact Table
fact_company_financials.to_csv(f'{export_folder}/Fact.csv', index=False)

print(f"Success! All {len(fact_cols)} columns mapped and 8 tables exported to the '{export_folder}' directory.")

Success! All 16 columns mapped and 8 tables exported to the 'Star_Schema_Export' directory.
